# 1. Reports, stations and soundings (`python` env)

NWS rainfall reports (PNS), flood Local Storm Reports, Area Forecast Discussions, ASOS hourly rain near the three rain cores, the Miami (KMFL) sounding PWAT climatology and kinematics, and the OISST downloads.

Cells are the **original session code, verbatim, in the order they ran** (only local paths in outputs were shortened). Each cell starts with a comment giving the original session cell number, environment and status. Failed cells are kept and marked, because the next cell is usually the fix.

**Outputs are the recorded outputs of the original run** (printed text, and each figure embedded in the last cell that saved it), attached from the run log, not produced by re-executing the notebook. Very long printouts are truncated with a note.

In [ ]:
# Setup (added for the repo): run from the project folder
import os
if os.path.basename(os.getcwd())=='notebooks': os.chdir('..')


In [ ]:
%%bash
# [orig cell 0 | env: python | status: ok]
for u in "https://mesonet.agron.iastate.edu/geojson/network.py?network=FL_ASOS" "https://noaa-mrms-pds.s3.amazonaws.com/?list-type=2&prefix=CONUS/MultiSensor_QPE_01H_Pass2_00.00/20261102/&max-keys=3" "https://nomads.ncep.noaa.gov/pub/data/nccf/com/gfs/prod/" "https://data.rcc-acis.org/StnMeta?state=FL&elems=pcpn" "https://noaa-gfs-bdp-pds.s3.amazonaws.com/?list-type=2&prefix=gfs.20261102/00/atmos/&max-keys=3"; do echo "$u -> $(curl -s -o /dev/null -w '%{http_code}' -m 20 "$u")"; done

https://mesonet.agron.iastate.edu/geojson/network.py?network=FL_ASOS -> 200
https://noaa-mrms-pds.s3.amazonaws.com/?list-type=2&prefix=CONUS/MultiSensor_QPE_01H_Pass2_00.00/20261102/&max-keys=3 -> 000
https://nomads.ncep.noaa.gov/pub/data/nccf/com/gfs/prod/ -> 000
https://data.rcc-acis.org/StnMeta?state=FL&elems=pcpn -> 000
https://noaa-gfs-bdp-pds.s3.amazonaws.com/?list-type=2&prefix=gfs.20261102/00/atmos/&max-keys=3 -> 000


In [ ]:
# [orig cell 1 | env: python | status: error (failed in the original run; kept for the record)]
import requests, pandas as pd, numpy as np, io, json, os
os.makedirs('event', exist_ok=True)
# Station metadata
g = requests.get("https://mesonet.agron.iastate.edu/geojson/network.py?network=FL_ASOS", timeout=60).json()
st = pd.DataFrame([{**f['properties'], 'lon':f['geometry']['coordinates'][0], 'lat':f['geometry']['coordinates'][1]} for f in g['features']])
def hav(lat1, lon1, lat2, lon2):
    R=6371; p=np.pi/180
    a=np.sin((lat2-lat1)*p/2)**2+np.cos(lat1*p)*np.cos(lat2*p)*np.sin((lon2-lon1)*p/2)**2
    return 2*R*np.arcsin(np.sqrt(a))
hot = {'Palm Beach Gardens':(26.823,-80.138),'Sunrise':(26.157,-80.286),'Melbourne':(28.083,-80.608)}
for k,(la,lo) in hot.items(): st['d_'+k]=hav(la,lo,st.lat,st.lon)
st['dmin']=st[[c for c in st if c.startswith('d_')]].min(axis=1)
near = st[st.dmin<70].sort_values('lat',ascending=False)
print(len(near)); print(near[['sid','name','lat','lon','dmin']].round(2).to_string())
st.to_csv('event/fl_asos_meta.csv',index=False)
# LSRs
lsr = requests.get("https://mesonet.agron.iastate.edu/geojson/lsr.php?sts=202610311200&ets=202611041200&wfos=MFL,MLB", timeout=60).json()
L = pd.DataFrame([f['properties'] for f in lsr['features']])
print(len(L), L.columns.tolist()); print(L['typetext'].value_counts().head(10))
L.to_csv('event/lsr_MFL_MLB.csv',index=False)


24


Traceback (most recent call last):
  File "<kernel:2>", line 15, in <module>
    print(len(near)); print(near[['sid','name','lat','lon','dmin']].round(2).to_string())
                            ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "~/miniconda3/envs/python/lib/python3.11/site-packages/pandas/core/frame.py", line 4119, in __getitem__
    indexer = self.columns._get_indexer_strict(key, "columns")[1]
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "~/miniconda3/envs/python/lib/python3.11/site-packages/pandas/core/indexes/base.py", line 6212, in _get_indexer_strict
    self._raise_if_missing(keyarr, indexer, axis_name)
  File "~/miniconda3/envs/python/lib/python3.11/site-packages/pandas/core/indexes/base.py", line 6264, in _raise_if_missing
    raise KeyError(f"{not_found} not in index")
KeyError: "['name'] not in index"


In [ ]:
# [orig cell 2 | env: python | status: ok]
print(st.columns.tolist()[:20])
nm = 'sname' if 'sname' in st else [c for c in st if 'name' in c.lower()][0]
print(near[['sid',nm,'lat','lon','dmin']].round(2).to_string())
lsr = requests.get("https://mesonet.agron.iastate.edu/geojson/lsr.php?sts=202610311200&ets=202611041200&wfos=MFL,MLB", timeout=60).json()
L = pd.DataFrame([f['properties'] for f in lsr['features']])
print(len(L), L.columns.tolist()); print(L['typetext'].value_counts().head(10))
L.to_csv('event/lsr_MFL_MLB.csv',index=False)


['elevation', 'sname', 'time_domain', 'archive_begin', 'archive_end', 'state', 'country', 'climate_site', 'wfo', 'tzname', 'ncdc81', 'ncei91', 'ugc_county', 'ugc_zone', 'county', 'sid', 'network', 'online', 'synop', 'attributes']
     sid                        sname    lat    lon   dmin
106  X21                   Titusville  28.62 -80.84  63.99
97   TTS           NASA SHUTTLE FCLTY  28.61 -80.69  59.75
92   TIX                   TITUSVILLE  28.51 -80.80  51.53
111  XMR               Cape Canaveral  28.47 -80.57  42.97
19   COI               Merritt Island  28.34 -80.69  29.74
18   COF          PATRICK AFB/COCOA B  28.24 -80.61  16.90
61   MLB           MELBOURNE REGIONAL  28.10 -80.65   4.27
108  X59                     Valkaria  27.96 -80.56  14.45
107  X26                 Valkaria AFS  27.81 -80.50  32.02
102  VRB                   Vero Beach  27.66 -80.42  51.07
37   FPR           FT PIERCE/ST LUCIE  27.50 -80.38  68.90
90   SUA  WITHAM FIELD AIRPORT/STUART  27.18 -80.22  40.52
31 

In [ ]:
# [orig cell 3 | env: python | status: ok]
sids = near.sid.tolist()
url = ("https://mesonet.agron.iastate.edu/cgi-bin/request/hourlyprecip.py?network=FL_ASOS&"
       + "&".join(f"station={s}" for s in sids) + "&year1=2026&month1=10&day1=31&year2=2026&month2=11&day2=4&lalo=1&st=1&tz=America/New_York")
r = requests.get(url, timeout=120); print(r.text[:300])
hp = pd.read_csv(io.StringIO(r.text))
print(hp.shape, hp.columns.tolist())


station,network,valid,precip_in,lat,lon,st
PMP,FL_ASOS,2026-10-31 00:00,0.0,26.24636,-80.11103,FL
F45,FL_ASOS,2026-10-31 00:00,0.0,26.8459167,-80.2223333,FL
DJT,FL_ASOS,2026-10-31 00:00,0.0,26.6847,-80.0994,FL
MIA,FL_ASOS,2026-10-31 00:00,0.0,25.78805,-80.31693,FL
BCT,FL_ASOS,2026-10-31 00:00,0.0,26
(1990, 7) ['station', 'network', 'valid', 'precip_in', 'lat', 'lon', 'st']


In [ ]:
# [orig cell 4 | env: python | status: ok]
hp['valid']=pd.to_datetime(hp.valid); hp['mm']=hp.precip_in.clip(lower=0)*25.4
P = hp.pivot_table(index='valid',columns='station',values='mm',aggfunc='sum').sort_index()
P = P.reindex(pd.date_range('2026-10-31','2026-11-04 23:00',freq='h'))
rows=[]
for s in P:
    x=P[s]
    d = x.groupby(x.index.date).sum(min_count=1)
    r6 = x.rolling(6,min_periods=4).sum()
    rows.append(dict(sid=s, n_hours=int(x.notna().sum()), tot_mm=x.sum(), max1h=x.max(), t_max1h=x.idxmax(), max6h=r6.max(), t_max6h=r6.idxmax(),
                     **{f'{k:%m-%d}':v for k,v in d.items()}))
T = pd.DataFrame(rows).merge(near[['sid','sname','lat','lon']],on='sid').sort_values('lat',ascending=False)
pd.set_option('display.width',250)
print(T.round(1).drop(columns=['lat','lon']).to_string(index=False))


sid  n_hours  tot_mm  max1h             t_max1h  max6h             t_max6h  10-31  11-01  11-02  11-03  11-04                       sname
X21       94   255.5   44.4 2026-11-01 23:00:00  117.9 2026-11-01 23:00:00    0.0  126.7  108.5   20.3    NaN                  Titusville
TTS       53    19.8   10.7 2026-11-02 20:00:00   16.5 2026-11-03 01:00:00    0.3    NaN   15.2    4.3    NaN          NASA SHUTTLE FCLTY
TIX       57     0.0    0.0 2026-10-31 06:00:00    0.0 2026-10-31 09:00:00    0.0    0.0    0.0    0.0    NaN                  TITUSVILLE
XMR       54    27.2   12.7 2026-11-02 20:00:00   19.0 2026-11-03 00:00:00    0.3    NaN   24.6    2.3    NaN              Cape Canaveral
COI       94     0.0    0.0 2026-10-31 00:00:00    0.0 2026-10-31 03:00:00    0.0    0.0    0.0    0.0    NaN              Merritt Island
COF       53    29.2   13.7 2026-11-02 12:00:00   23.4 2026-11-02 17:00:00    3.6    NaN   25.7    0.0    NaN         PATRICK AFB/COCOA B
MLB       94   362.2   31.8 2026-1

In [ ]:
# [orig cell 5 | env: python | status: ok]
txt={}
for pil in ['PNSMFL','PNSMLB','RRMMFL','FFSMFL','FFWMFL','FFWMLB','FFSMLB']:
    u=f"https://mesonet.agron.iastate.edu/cgi-bin/afos/retrieve.py?pil={pil}&fmt=text&sdate=2026-11-01&edate=2026-11-05&limit=60"
    t=requests.get(u,timeout=60).text; txt[pil]=t
    open(f'event/{pil}.txt','w').write(t)
    print(pil, len(t), t.count('\x01') or t.count('NATIONAL WEATHER SERVICE'))
import re
# list PNS headlines
for pil in ['PNSMFL','PNSMLB']:
    for m in re.finditer(r'\.\.\.(.*?)\.\.\.', txt[pil][:200000]):
        s=m.group(1)
        if len(s)<120 and ('RAIN' in s.upper() or 'TOTAL' in s.upper()): print(pil,'|',s)


PNSMFL 224443 4
PNSMLB 58964 2
RRMMFL 43455 13
FFSMFL 3095 3
FFWMFL 23316 13
FFWMLB 45459 26
FFSMLB 5349 4


In [ ]:
%%bash
# [orig cell 6 | env: python | status: ok]
cd event; grep -n -i -E "public information statement|rainfall|storm total|^[0-9]{3,4} (AM|PM)" PNSMFL.txt | head -40; echo ----; grep -n -i -E "public information statement|rainfall|storm total" PNSMLB.txt | head -20; echo; sed -n 1,60p PNSMFL.txt | cut -c1-120

7:Public Information Statement
9:1059 AM EST Tue Nov 3 2026
318:11/03/2026, 0700 AM, FL, Broward, Coral Glades HS, , , 26.26361, -80.29667, RAIN, 2, Inch, EARTH-NETWORKS, Storm total rainfall,
319:11/03/2026, 0600 AM, FL, Broward, HOLLYWOOD, , , 26.0281, -80.1341, RAIN, 2.01, Inch, COOPC, Storm total rainfall,
320:11/03/2026, 0700 AM, FL, Broward, JetBlue Airways Corp, , , 26.07111, -80.14528, RAIN, 2.06, Inch, EARTH-NETWORKS, Storm total rainfall,
321:11/03/2026, 0700 AM, FL, Broward, K C Wright Admin Building, , , 26.11472, -80.14056, RAIN, 2.28, Inch, EARTH-NETWORKS, Storm total rainfall,
322:11/03/2026, 0600 AM, FL, Broward, FORT LAUDERDALE BEACH, , , 26.1408, -80.1061, RAIN, 2.43, Inch, COOPA, Storm total rainfall,
323:11/03/2026, 0655 AM, FL, Broward, DW6169 Hollywood, , , 26.0045, -80.1545, RAIN, 2.44, Inch, APRSWXNET/CWOP, Storm total rainfall,
324:11/03/2026, 0700 AM, FL, Broward, Henry D. Perry Education Ctr, , , 25.97944, -80.22583, RAIN, 2.49, Inch, EARTH-NETWORKS, Storm to

In [ ]:
# [orig cell 7 | env: python | status: ok]
pat = re.compile(r'^(.{3,29}?)\s+([\d.]+) Inch\s+(\d{4} [AP]M \d\d/\d\d)\s+([\d.]+)N/([\d.]+)W', re.M)
recs=[]
for pil in ['PNSMFL','PNSMLB']:
    for prod in txt[pil].split('\x01'):
        hdr = re.search(r'(\d{3,4} [AP]M E[SD]T \w+ \w+ \d+ 2026)', prod)
        title = re.search(r'\.\.\.(.*?(PRECIP|RAIN).*?)\.\.\.', prod)
        county=None
        for line in prod.splitlines():
            mc = re.match(r'^\.\.\.(.+) County\.\.\.', line)
            if mc: county=mc.group(1); continue
            m = pat.match(line)
            if m: recs.append(dict(wfo=pil[-3:], issued=hdr.group(1) if hdr else None, title=title.group(1) if title else None,
                    county=county, loc=m.group(1).strip(), inch=float(m.group(2)), obs=m.group(3), lat=float(m.group(4)), lon=-float(m.group(5))))
R = pd.DataFrame(recs); print(len(R)); print(R.groupby(['wfo','issued','title']).size())


518
wfo  issued                      title                                                            
MFL  1059 AM EST Tue Nov 3 2026  72 HOUR PRELIMINARY PRECIPITATION REPORTS                            272
MLB  113 PM EST Tue Nov 3 2026   PRECIPITATION REPORTS FOR 12 AM SUN NOV 1 THROUGH 12 PM TUE NOV 3    157
     338 AM EST Tue Nov 3 2026   PRECIPITATION REPORTS                                                 89
dtype: int64


In [ ]:
# [orig cell 8 | env: python | status: ok]
Rm = R[R.issued.str.startswith('1059')|R.issued.str.startswith('113 PM')]
print(Rm.sort_values('inch',ascending=False).head(25)[['wfo','county','loc','inch','obs','lat','lon']].to_string(index=False))
print(Rm.groupby('county').inch.agg(['max','median','size']).sort_values('max',ascending=False).head(12))
# other MFL products headers
for prod in txt['PNSMFL'].split('\x01'):
    h=re.findall(r'\.\.\.([A-Z0-9 ,/-]{8,90})\.\.\.',prod)[:3]; print(h, len(prod))


wfo       county                          loc  inch           obs   lat    lon
MLB      Brevard           1 N Grant-Valkaria 24.69 1140 AM 11/03 27.96 -80.56
MLB      Brevard                  2 E Malabar 24.52 1130 AM 11/03 27.98 -80.56
MFL   Palm Beach            3 W Riviera Beach 23.19 0700 AM 11/03 26.78 -80.13
MLB         None          2 NE Grant-Valkaria 22.75 1100 AM 11/03 27.98 -80.54
MLB      Brevard                 1 S Palm Bay 22.00 1135 AM 11/03 27.97 -80.66
MLB         None          1 S Melbourne Beach 20.87 1138 AM 11/03 28.05 -80.56
MLB      Brevard            2 E Bayside Lakes 20.33 0700 AM 11/03 27.95 -80.64
MLB      Brevard            1 E Bayside Lakes 20.32 0700 AM 11/03 27.95 -80.64
MLB      Brevard                 3 W Palm Bay 19.68 0900 AM 11/03 27.99 -80.70
MLB      Brevard                2 NE Palm Bay 19.46 0645 AM 11/03 28.01 -80.65
MLB      Brevard                 3 NW Malabar 19.30 0700 AM 11/03 28.02 -80.62
MFL   Palm Beach             2 NE The Acreage 19.27 

In [ ]:
# [orig cell 9 | env: python | status: ok]
cores = {'N Palm Beach (3 W Riviera Beach)':(26.78,-80.13),'W Broward (2 W Lauderhill)':(26.15,-80.24),'S Brevard (1 N Grant-Valkaria)':(27.96,-80.56)}
for k,(la,lo) in cores.items():
    d = st.assign(km=hav(la,lo,st.lat,st.lon)).nsmallest(5,'km')
    print(k, '|', ', '.join(f"{r.sid} {r.km:.1f} km" for r in d.itertuples()))
print(P.loc['2026-11-01 06':'2026-11-03 06',['DJT','F45','FXE','PMP','HWO','X26','MLB']].dropna(how='all').query('DJT>5 or FXE>5 or PMP>5 or X26>5 or MLB>5').round(1).to_string())
print(L[['valid','typetext','city','county','remark']].assign(remark=L.remark.str[:110]).sort_values('valid').to_string(index=False))


N Palm Beach (3 W Riviera Beach) | DJT 11.0 km, F45 11.7 km, LNA 21.3 km, BCT 44.5 km, SUA 45.4 km
W Broward (2 W Lauderhill) | FXE 8.9 km, FLL 11.1 km, HWO 16.7 km, PMP 16.7 km, OPF 27.0 km
S Brevard (1 N Grant-Valkaria) | X59 0.2 km, X26 17.6 km, MLB 18.0 km, COF 31.0 km, VRB 36.6 km
station               DJT  F45   FXE   PMP   HWO   X26   MLB
2026-11-01 07:00:00   0.0  0.0   5.8   0.8   3.6   0.0   0.0
2026-11-01 13:00:00   5.1  0.0   2.0   1.0   7.9   1.3   0.0
2026-11-01 14:00:00   6.4  0.3  33.0  20.1   3.3   2.8  12.4
2026-11-01 15:00:00   6.1  0.0   1.8   1.8   1.3   2.8   3.6
2026-11-01 16:00:00  37.1  0.0   2.8   6.1   2.8   0.3   2.3
2026-11-01 17:00:00  31.0  0.0   0.0   0.0   0.0   3.6   0.5
2026-11-01 18:00:00   1.3  0.0   0.0   0.0   0.0  10.4   1.0
2026-11-01 19:00:00  20.1  0.3   0.0   0.0   0.3  16.0   2.3
2026-11-01 20:00:00   0.5  NaN   0.3   0.0   3.3   2.0   6.9
2026-11-01 21:00:00   0.0  2.0   0.8   0.0   1.3   0.3   5.8
2026-11-01 23:00:00   0.0  3.0   0.0   0.0

In [ ]:
# [orig cell 10 | env: python | status: ok]
afd={}
for pil in ['AFDMFL','AFDMLB']:
    t=requests.get(f"https://mesonet.agron.iastate.edu/cgi-bin/afos/retrieve.py?pil={pil}&fmt=text&sdate=2026-10-31&edate=2026-11-04&limit=40",timeout=60).text
    afd[pil]=t; open(f'event/{pil}.txt','w').write(t)
    prods=[p for p in t.split('\x01') if len(p)>500]; print(pil,len(prods))
kw = re.compile(r'(PWAT|precipitable|inch(es)? of PW|low[- ]level jet|mid-level low|upper[- ]level low|upper low|trough|front|easterly|convergence|training|backbuilding|back-building|moisture|record|percentile|sea surface|warm sst|coastal trough)', re.I)
# print key sentences from the MFL AFDs issued Nov 1 – Nov 2
for pil in ['AFDMFL','AFDMLB']:
    for p in [p for p in afd[pil].split('\x01') if len(p)>500]:
        hdr=re.search(r'\d{3,4} [AP]M E[SD]T \w+ \w+ \d+ 2026',p)
        if not hdr or not re.search(r'(Sat Oct 31|Sun Nov 1|Mon Nov 2) ',hdr.group(0)): continue
        body=re.sub(r'\s+',' ',p)
        sents=[s for s in re.split(r'(?<=\.) ',body) if kw.search(s) and 40<len(s)<400]
        print('\n##',pil,hdr.group(0)); print('\n'.join('- '+s for s in sents[:7]))


AFDMFL 22
AFDMLB 22

## AFDMFL 1259 PM EST Mon Nov 2 2026
- Issued at 1202 PM EST Mon Nov 2 2026 The aforementioned shortwave across the eastern Gulf continues to enhance upper level divergence across South Florida early this afternoon, enhancing low level convergence and facilitating a gradual spatial expansion of shower and thunderstorm activity across the region.
- There's very little surface reflection to this upper low, with the latest WPC surface analysis depicting high pressure centered over the Mid-Atlantic/NW Atlantic, with its western reaches covering much of the eastern CONUS.
- Locally, we remain under easterly flow at the base of the ridge with a coastal trough still established down the east coast of Florida.
- They also depict convection in the SE Gulf, associated with the upper low, moving across south Florida through the afternoon and evening.
- As the base of the elongated trough cuts off over the SE Gulf today, it will be slow to move out of the region and likely hav

In [ ]:
# [orig cell 17 | env: python | status: ok]
Rm.to_csv('event/pns_72h.csv',index=False); T.to_csv('event/asos_event_summary.csv',index=False); P.to_csv('event/asos_hourly_mm_local.csv')
# OISST daily (preliminary) files
oi={}
for d in ['20261030','20261031','20261101']:
    for suf in ['_preliminary','']:
        u=f"https://www.ncei.noaa.gov/data/sea-surface-temperature-optimum-interpolation/v2.1/access/avhrr/{d[:6]}/oisst-avhrr-v02r01.{d}{suf}.nc"
        r=requests.get(u,timeout=120)
        if r.status_code==200: open(f'event/oisst_{d}.nc','wb').write(r.content); oi[d]=suf; break
print(oi)
# KMFL raob: IEM bulk request
u="https://mesonet.agron.iastate.edu/cgi-bin/request/raob.py?station=KMFL&sts=2026-10-30T00:00Z&ets=2026-11-03T13:00Z"
r=requests.get(u,timeout=120); print(r.status_code, r.text[:400])


{'20261030': '_preliminary', '20261031': '_preliminary', '20261101': '_preliminary'}
200 station,validUTC,levelcode,pressure_mb,height_m,tmpc,dwpc,drct,speed_kts,bearing,range_sm
KMFL,2026-10-30 00:00:00,4,1013.0,5.0,27.8,19.8,50.0,8,M,M
KMFL,2026-10-30 00:00:00,4,1012.0,13.57,27.8,22.8,M,M,M,M
KMFL,2026-10-30 00:00:00,4,1000.0,117.0,26.8,21.8,30.0,12,M,M
KMFL,2026-10-30 00:00:00,4,978.83,305.0,25.16,20.98,35.0,17,M,M
KMFL,2026-10-30 00:00:00,4,945.44,610.0,22.49,19.64,25.0,17,M,M
KMF


In [ ]:
# [orig cell 18 | env: python | status: error (failed in the original run; kept for the record)]
from concurrent.futures import ThreadPoolExecutor
def fetch(y):
    u=f"https://mesonet.agron.iastate.edu/cgi-bin/request/raob.py?station=KMFL&sts={y}-10-15T00:00Z&ets={y}-11-16T00:00Z"
    return requests.get(u,timeout=300).text
with ThreadPoolExecutor(6) as ex: texts=list(ex.map(fetch,range(1991,2026)))
texts.append(r.text)
S=pd.concat([pd.read_csv(io.StringIO(t),na_values='M') for t in texts if t.startswith('station')])
S['validUTC']=pd.to_datetime(S.validUTC); S=S.dropna(subset=['pressure_mb'])
def pw(g):
    g=g.dropna(subset=['dwpc']).sort_values('pressure_mb',ascending=False)
    g=g[g.pressure_mb>=300]
    if len(g)<8 or g.pressure_mb.max()<950 or g.pressure_mb.min()>350: return np.nan
    e=6.112*np.exp(17.67*g.dwpc/(g.dwpc+243.5)); p=g.pressure_mb.values
    q=0.622*e/(p-0.378*e)
    return np.trapz(q[::-1],p[::-1]*100)/9.81  # kg/m2 = mm
PW=S.groupby('validUTC').apply(pw).dropna().rename('pw_mm').to_frame()
PW['year']=PW.index.year
clim=PW[(PW.year<=2025)]
ev=PW[PW.index>='2026-10-30']
print(len(clim), clim.year.nunique())
print(clim.pw_mm.describe(percentiles=[.5,.9,.95,.99]).round(1))
ev=ev.assign(pctile=[(clim.pw_mm<v).mean()*100 for v in ev.pw_mm], rank_of=len(clim)+1,
             rank=[int((clim.pw_mm>=v).sum())+1 for v in ev.pw_mm])
print(ev.round(1))
PW.to_csv('event/kmfl_pwat.csv')


<kernel:12>:8: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  S=pd.concat([pd.read_csv(io.StringIO(t),na_values='M') for t in texts if t.startswith('station')])
Traceback (most recent call last):
  File "<kernel:12>", line 17, in <module>
    PW=S.groupby('validUTC').apply(pw).dropna().rename('pw_mm').to_frame()
       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "~/miniconda3/envs/python/lib/python3.11/site-packages/pandas/core/groupby/groupby.py", line 1826, in apply
    result = self._python_apply_general(f, self._selected_obj)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "~/miniconda3/envs/python/lib/python3.11/site-packages/pandas/core/groupby/groupby.py", line 1887, in _python_apply_general
    values, mutated = self

In [ ]:
# [orig cell 19 | env: python | status: ok]
def pw(g):
    g=g.dropna(subset=['dwpc']).drop_duplicates('pressure_mb').sort_values('pressure_mb',ascending=False)
    g=g[g.pressure_mb>=300]
    if len(g)<8 or g.pressure_mb.max()<950 or g.pressure_mb.min()>350: return np.nan
    e=6.112*np.exp(17.67*g.dwpc/(g.dwpc+243.5)); p=g.pressure_mb.values
    q=(0.622*e/(p-0.378*e)).values
    return -np.trapezoid(q,p*100)/9.81
PW=S.groupby('validUTC').apply(pw).dropna().rename('pw_mm').to_frame(); PW['year']=PW.index.year
clim=PW[PW.year<=2025]; ev=PW[PW.index>='2026-10-30'].copy()
print(len(clim), clim.year.nunique(), clim.pw_mm.quantile([.5,.9,.95,.99]).round(1).to_dict(), clim.pw_mm.max().round(1), clim.pw_mm.idxmax())
ev['pctile']=[(clim.pw_mm<v).mean()*100 for v in ev.pw_mm]; ev['rank']=[int((clim.pw_mm>=v).sum())+1 for v in ev.pw_mm]
print(ev.round(1).to_string()); PW.to_csv('event/kmfl_pwat.csv')


1989 31 {0.5: 34.5, 0.9: 52.3, 0.95: 56.1, 0.99: 61.4} 67.2 2011-10-19 12:00:00
                     pw_mm  year  pctile  rank
validUTC                                      
2026-10-30 00:00:00   43.5  2026    72.5   548
2026-10-30 12:00:00   40.9  2026    66.1   675
2026-10-31 00:00:00   43.6  2026    73.0   539
2026-10-31 12:00:00   42.8  2026    70.3   591
2026-11-01 00:00:00   45.6  2026    77.9   441
2026-11-01 12:00:00   53.6  2026    91.6   168
2026-11-02 00:00:00   53.0  2026    90.6   187
2026-11-02 12:00:00   46.1  2026    79.0   419
2026-11-03 00:00:00   55.9  2026    94.8   105
2026-11-03 12:00:00   50.9  2026    87.7   246


<kernel:13>:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  PW=S.groupby('validUTC').apply(pw).dropna().rename('pw_mm').to_frame(); PW['year']=PW.index.year

In [ ]:
# [orig cell 20 | env: python | status: error (failed in the original run; kept for the record)]
import xarray as xr
nov = clim[(clim.index.month==11)]
ev['pct_Nov1-15']=[(nov.pw_mm<v).mean()*100 for v in ev.pw_mm]
print('Nov1-15 clim n=',len(nov), nov.pw_mm.quantile([.5,.9,.99]).round(1).to_dict()); print(ev[['pw_mm','pct_Nov1-15']].round(1).loc['2026-11-01':].to_string())
def kin(t):
    g=S[S.validUTC==t].dropna(subset=['pressure_mb']).drop_duplicates('pressure_mb').sort_values('pressure_mb',ascending=False)
    w=g.dropna(subset=['drct','speed_kts'])
    u=-w.speed_kts*0.5144*np.sin(np.radians(w.drct)); v=-w.speed_kts*0.5144*np.cos(np.radians(w.drct)); p=w.pressure_mb
    def lay(a,b): m=(p<=a)&(p>=b); return np.array([u[m].mean(),v[m].mean()])
    mw=lay(850,300); llj=lay(925,850); cor=mw-llj   # Corfidi upwind propagation (MBE) vector
    z0=np.interp(0,g.tmpc.dropna().values[::-1],g.height_m[g.tmpc.notna()].values[::-1])
    sfc=g.iloc[0]; lcl=125*(sfc.tmpc-sfc.dwpc)
    def spd(x): return np.hypot(*x)
    def dirn(x): return (np.degrees(np.arctan2(-x[0],-x[1]))%360)
    return dict(time=t, mean_wind_850_300_ms=spd(mw), mean_wind_dir=dirn(mw), llj_925_850_ms=spd(llj), llj_dir=dirn(llj),
                corfidi_upwind_ms=spd(cor), freezing_lvl_m=z0, lcl_m=lcl, warm_cloud_depth_m=z0-lcl)
K=pd.DataFrame([kin(t) for t in pd.to_datetime(['2026-11-01 00:00','2026-11-01 12:00','2026-11-02 00:00','2026-11-02 12:00','2026-11-03 00:00'])])
print(K.round(1).to_string(index=False))
# OISST anomaly off SE Florida
for d in ['20261030','20261031','20261101']:
    o=xr.open_dataset(f'event/oisst_{d}.nc').squeeze()
    box=lambda la0,la1,lo0,lo1: o.sel(lat=slice(la0,la1),lon=slice(360+lo0,360+lo1))
    a=box(25.5,28.5,-80.0,-78.5); b=box(24,32,-80,-70); c=box(22,26,-84,-80)
    print(d, 'Gulf Stream E of SE FL: SST %.2f anom %+.2f | W Atlantic 24-32N,80-70W: anom %+.2f (max %+.2f) | Straits/SE Gulf anom %+.2f' % (
        float(a.sst.mean()),float(a.anom.mean()),float(b.anom.mean()),float(b.anom.max()),float(c.anom.mean())))


Traceback (most recent call last):
  File "<kernel:14>", line 2, in <module>
    import xarray as xr
ModuleNotFoundError: No module named 'xarray'


In [ ]:
# [orig cell 21 | env: python | status: ok]
nov = clim[(clim.index.month==11)]
ev['pct_Nov1-15']=[(nov.pw_mm<v).mean()*100 for v in ev.pw_mm]
print('Nov1-15 clim n=',len(nov), nov.pw_mm.quantile([.5,.9,.99]).round(1).to_dict()); print(ev[['pw_mm','pct_Nov1-15']].round(1).loc['2026-11-01':].to_string())
def kin(t):
    g=S[S.validUTC==t].dropna(subset=['pressure_mb']).drop_duplicates('pressure_mb').sort_values('pressure_mb',ascending=False)
    w=g.dropna(subset=['drct','speed_kts'])
    u=-w.speed_kts*0.5144*np.sin(np.radians(w.drct)); v=-w.speed_kts*0.5144*np.cos(np.radians(w.drct)); p=w.pressure_mb
    def lay(a,b): m=(p<=a)&(p>=b); return np.array([u[m].mean(),v[m].mean()])
    mw=lay(850,300); llj=lay(925,850); cor=mw-llj
    gt=g.dropna(subset=['tmpc']); z0=np.interp(0,gt.tmpc.values[::-1],gt.height_m.values[::-1])
    sfc=g.dropna(subset=['tmpc','dwpc']).iloc[0]; lcl=125*(sfc.tmpc-sfc.dwpc)
    spd=lambda x: np.hypot(*x); dirn=lambda x: np.degrees(np.arctan2(-x[0],-x[1]))%360
    return dict(time=t, mw850_300=spd(mw), mw_dir=dirn(mw), w925_850=spd(llj), w925_850_dir=dirn(llj),
                corfidi_upwind=spd(cor), frz_m=z0, lcl_m=lcl, warm_cloud_m=z0-lcl, sfc_T=sfc.tmpc, sfc_Td=sfc.dwpc)
K=pd.DataFrame([kin(t) for t in pd.to_datetime(['2026-11-01 00:00','2026-11-01 12:00','2026-11-02 00:00','2026-11-02 12:00','2026-11-03 00:00'])])
print(K.round(1).to_string(index=False)); K.to_csv('event/kmfl_kinematics.csv',index=False)


Nov1-15 clim n= 927 {0.5: 31.6, 0.9: 47.8, 0.99: 59.6}
                     pw_mm  pct_Nov1-15
validUTC                               
2026-11-01 00:00:00   45.6         86.1
2026-11-01 12:00:00   53.6         96.5
2026-11-02 00:00:00   53.0         95.8
2026-11-02 12:00:00   46.1         87.4
2026-11-03 00:00:00   55.9         97.7
2026-11-03 12:00:00   50.9         94.1
               time  mw850_300  mw_dir  w925_850  w925_850_dir  corfidi_upwind  frz_m  lcl_m  warm_cloud_m  sfc_T  sfc_Td
2026-11-01 00:00:00        2.0   240.0       9.0          76.4            10.9 4995.3  362.5        4632.8   27.8    24.9
2026-11-01 12:00:00        3.1   246.8       7.5         106.4            10.0 4780.6  487.5        4293.1   26.0    22.1
2026-11-02 00:00:00        3.8   182.4       7.2          90.9             8.2 4171.0  262.5        3908.5   24.4    22.3
2026-11-02 12:00:00        5.7   200.2      11.7         123.6            11.7 4532.6  487.5        4045.1   26.0    22.1
2026-11-03 00:0

In [ ]:
# [orig cell 31 | env: python | status: ok]
st.to_csv('event/fl_asos_meta.csv',index=False); print(os.path.abspath('event/fl_asos_meta.csv'))

./event/fl_asos_meta.csv
